# Ders 9: Geometrik Derin Öğrenme — Simetri, İfade Gücü ve Çizgeler

**İleri Derin Öğrenme** — Haydar Kılıç

Ön koşul: *Derin Öğrenme*, Ders 6 ve 11 (ESA'lar, Çizge Sinir Ağları).

Geometrik derin öğrenme, ESA'ların, ÇSA'ların, transformer'ların ve derin kümelerin (deep sets) aynı
kurgunun farklı **simetri gruplarına** uygulanmış hâlleri olduğu gözleminden ibarettir. Bu ciddiye
alındığında iki soru keskinleşir: bir simetriyi mimariye gömmek bize ne kazandırır ve ortaya çıkan
modelin *ifade gücü tavanı* nedir? Mesaj geçişli ÇSA'lar için ikinci sorunun cevabı tam olarak
bilinir ve biraz hayal kırıklığı yaratır — 1-Weisfeiler–Lehman renk arıtma testinden daha güçlü
değillerdir.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from itertools import permutations

np.random.seed(0)
plt.rcParams["figure.dpi"] = 100
print("Kütüphaneler yüklendi.")


## 1. Değişmezlik, Eşdeğerlik ve Kurgu

Girdi üzerinde etki eden bir $G$ grubu için bir $f$ fonksiyonu

- $f(g \cdot x) = f(x)$ ise **değişmezdir** (invariant) — çıktı dönüşümü yok sayar (görüntü → sınıf
  etiketi),
- $f(g \cdot x) = g \cdot f(x)$ ise **eşdeğerlidir** (equivariant) — çıktı da birlikte dönüşür
  (görüntü → bölütleme maskesi).

Bu defterdeki her mimarinin kurgusu aynıdır: **eşdeğerli** katmanları üst üste yığ, sonda bir kez
**değişmez** havuzlama uygula. Eşdeğerliği ağın içinde korumak, çok erken havuzlamayla yok edilecek
uzamsal veya ilişkisel bilgiyi saklar.

| Mimari | Alan | Simetri grubu |
|---|---|---|
| MLP | vektörler | yok |
| ESA | ızgara | ötelemeler (yönlendirilebilir ESA'larda + dönmeler) |
| Deep Sets | kümeler | permütasyonlar $S_n$ |
| ÇSA | çizgeler | komşuluğa saygılı permütasyonlar |
| Transformer | dikkatli kümeler | permütasyonlar (konum kodlamaları bunu kasten bozar) |
| Küresel ESA | küre | $SO(3)$ |


In [ ]:
# Öteleme altında evrişimin ve düz bir MLP katmanının eşdeğerliği
n = 40
x = np.zeros(n); x[12:18] = 1.0; x[25] = -0.8
k = np.array([0.25, 0.5, 0.25])

def conv1d(sig, ker):
    return np.array([np.dot(np.roll(sig, -i)[:len(ker)], ker) for i in range(len(sig))])

W_mlp = np.random.randn(n, n)*0.15
shift = 7
roll = lambda v, s: np.roll(v, s)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
axes[0].plot(x, lw=2, label="x"); axes[0].plot(roll(x, shift), lw=2, ls="--", label=f"ötelenmiş x")
axes[0].set_title("Girdi ve ötelenmiş hâli"); axes[0].legend(fontsize=9); axes[0].grid(alpha=0.3)

axes[1].plot(conv1d(roll(x, shift), k), lw=2, label="evrişim(öteleme(x))")
axes[1].plot(roll(conv1d(x, k), shift), lw=2, ls="--", label="öteleme(evrişim(x))")
axes[1].set_title(f"Evrişim: eşdeğerli  (maks fark {np.abs(conv1d(roll(x,shift),k)-roll(conv1d(x,k),shift)).max():.1e})")
axes[1].legend(fontsize=9); axes[1].grid(alpha=0.3)

axes[2].plot(W_mlp @ roll(x, shift), lw=2, label="W öteleme(x)")
axes[2].plot(roll(W_mlp @ x, shift), lw=2, ls="--", label="öteleme(W x)")
axes[2].set_title("Tam bağlı katman: eşdeğerli değil")
axes[2].legend(fontsize=9); axes[2].grid(alpha=0.3)
plt.tight_layout(); plt.show()

# Simetrinin kazandırdığı: etkin örneklem büyüklüğü
group_size = np.array([1, 2, 4, 8, 24, 48, 100, 1000])
plt.figure(figsize=(7.5, 3.8))
plt.loglog(group_size, 1/np.sqrt(group_size), "o-", lw=2, label="genelleme açığı ~ 1/sqrt(|G|)")
plt.xlabel("simetri grubunun büyüklüğü |G|"); plt.ylabel("göreli hata (temsili)")
plt.title("Bir simetriyi mimariye gömmek, veri kümesini |G| ile çarpmak gibidir")
plt.grid(alpha=0.3, which="both"); plt.legend(fontsize=9); plt.tight_layout(); plt.show()


## 2. Veri Artırımına Karşı Mimariye Gömülü Eşdeğerlik

Veri artırımı ve mimari eşdeğerlik aynı değişmezliği hedefler ama eşdeğer değildir. Artırım
*öğrenilen fonksiyonu* veri dağılımı üzerinde yaklaşık değişmez yapar; eşdeğerlik ise dağılım dışı
noktalar dahil **her yerde tam** değişmez yapar ve gereksiz kopyaları temsil etmeye harcanacak
parametreleri ortadan kaldırır.

Aşağıdaki karşılaştırmada permütasyona-değişmez bir hedefe üç farklı yolla doğrusal model
uyduruluyor: ham, artırılmış ve değişmezliği özniteliklere gömülmüş (toplam havuzlama) hâliyle.


In [ ]:
d = 8
rng = np.random.default_rng(1)
target = lambda X: (X.sum(1)**2)/d + X.min(1)          # permütasyona değişmez

def features_raw(X):  return np.c_[X, X**2]
def features_inv(X):  return np.c_[X.sum(1), (X**2).sum(1), X.min(1), X.max(1)]   # simetrik fonksiyonlar

def fit_eval(feat, n_train, augment=0, seed=0):
    r = np.random.default_rng(seed)
    Xtr = r.normal(size=(n_train, d))
    if augment:
        Xtr = np.concatenate([r.permutation(Xtr, axis=1) for _ in range(augment)] + [Xtr])
    ytr = target(Xtr)
    Xte = r.normal(size=(3000, d)); yte = target(Xte)
    A = feat(Xtr); B = feat(Xte)
    w = np.linalg.solve(A.T@A + 1e-6*np.eye(A.shape[1]), A.T@ytr)
    return np.mean((B@w - yte)**2)/np.var(yte)

sizes = [20, 40, 80, 160, 320, 640, 1280]
curves = {
    "ham öznitelikler":                 [fit_eval(features_raw, m) for m in sizes],
    "ham + permütasyon artırımı x8": [fit_eval(features_raw, m, augment=8) for m in sizes],
    "değişmez öznitelikler":           [fit_eval(features_inv, m) for m in sizes],
}

fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
for name, c in curves.items():
    axes[0].loglog(sizes, c, "o-", lw=2, label=name)
axes[0].set_xlabel("training examples"); axes[0].set_ylabel("göreli test MSE")
axes[0].set_title("Aynı değişmezlik, onu elde etmenin üç yolu")
axes[0].legend(fontsize=9); axes[0].grid(alpha=0.3, which="both")

# Kesinlik: test girdisi permüte edilerek her modelin ne kadar değişmez olduğu ölçülüyor
def invariance_error(feat, n_train=320, seed=0, augment=0):
    r = np.random.default_rng(seed)
    Xtr = r.normal(size=(n_train, d))
    if augment:
        Xtr = np.concatenate([r.permutation(Xtr, axis=1) for _ in range(augment)] + [Xtr])
    ytr = target(Xtr); A = feat(Xtr)
    w = np.linalg.solve(A.T@A + 1e-6*np.eye(A.shape[1]), A.T@ytr)
    Xte = r.normal(size=(500, d))
    p0 = feat(Xte)@w
    errs = [np.abs(feat(r.permutation(Xte, axis=1))@w - p0).mean() for _ in range(20)]
    return np.mean(errs)

names = ["ham", "ham + artırım", "değişmez"]
vals  = [invariance_error(features_raw), invariance_error(features_raw, augment=8),
         invariance_error(features_inv)]
axes[1].bar(names, vals, color=["#c6dbef", "#6baed6", "#08519c"])
axes[1].set_yscale("log"); axes[1].set_ylabel("ortalama |f(perm(x)) - f(x)|  (log)")
axes[1].set_title("Artırım yaklaşır; mimari garanti eder")
plt.tight_layout(); plt.show()
for n_, v in zip(names, vals):
    print(f"{n_:16s} değişmezlik ihlali {v:.2e}")


## 3. Mesaj Geçişi ve Permütasyon Eşdeğerliği

Bir mesaj geçişli ÇSA katmanı şudur:

$$h_v^{(k+1)} = \phi\Big(h_v^{(k)},\ \bigoplus_{u \in \mathcal{N}(v)} \psi\big(h_v^{(k)}, h_u^{(k)}\big)\Big),$$

burada $\bigoplus$ **permütasyona-değişmez** bir toplayıcıdır (toplam, ortalama, maksimum). Toplayıcı
seçimi kozmetik değildir — katmanın neyi ölçebileceğini belirler:

| Toplayıcı | Ayırt edebildiği | Ayırt edemediği |
|---|---|---|
| **toplam** | çoklu kümeler (sayılar ve değerler) | — (üçünün en ifade gücü yükseği) |
| **ortalama** | oranlar | $\{a\}$ ile $\{a,a\}$ |
| **maksimum** | farklı değerlerin kümesi | $\{a,b\}$ ile $\{a,b,b\}$ |

GIN'in toplam toplayıcı ve ardından bir MLP kullanmasının nedeni tam olarak budur: toplam, çoklu
kümeler üzerinde birebir olan seçimdir ve ifade gücü ispatının ihtiyaç duyduğu şey birebirliktir.


In [ ]:
multisets = {"{a}": [1.0], "{a,a}": [1.0, 1.0], "{a,b}": [1.0, 2.0], "{a,b,b}": [1.0, 2.0, 2.0]}
aggs = {"toplam": np.sum, "ortalama": np.mean, "maks": np.max}

print(f"{'çoklu küme':12s}" + "".join(f"{a:>8s}" for a in aggs))
table = []
for name, s in multisets.items():
    row = [f(np.array(s)) for f in aggs.values()]
    table.append(row)
    print(f"{name:10s}" + "".join(f"{v:8.2f}" for v in row))
table = np.array(table)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
labels = list(multisets)
for i, a in enumerate(aggs):
    axes[0].plot(labels, table[:, i], "o-", lw=2, label=a)
axes[0].set_ylabel("toplanmış değer"); axes[0].set_title("Çakışmalar: aynı değerler = ayırt edilemez")
axes[0].legend(fontsize=9); axes[0].grid(alpha=0.3)

# Bir mesaj geçişi katmanının permütasyon eşdeğerliği, sayısal olarak doğrulanıyor
def mp_layer(A, H, W1, W2):
    return np.tanh(H@W1 + A@H@W2)                     # komşular üzerinde toplam toplayıcı

n = 6
A = (np.random.rand(n, n) < 0.4).astype(float); A = np.triu(A, 1); A = A + A.T
H = np.random.randn(n, 4); W1, W2 = np.random.randn(4, 4), np.random.randn(4, 4)
P = np.eye(n)[np.random.permutation(n)]
lhs = mp_layer(P@A@P.T, P@H, W1, W2)
rhs = P @ mp_layer(A, H, W1, W2)
print(f"\nmaks |f(PAP^T, PH) - P f(A,H)| = {np.abs(lhs-rhs).max():.2e}   -> permütasyona eşdeğerli")

axes[1].imshow(np.hstack([lhs, rhs]), cmap="RdBu_r")
axes[1].axvline(3.5, c="k", lw=2)
axes[1].set_title("f(permüte girdi)  |  permüte f(girdi)"); axes[1].set_xticks([]); axes[1].set_yticks([])
plt.tight_layout(); plt.show()


## 4. İfade Gücü Tavanı: 1-WL

**Weisfeiler–Lehman renk arıtma**, her düğümü kendi rengiyle komşularının renk çoklu kümesinin bir
özet değeriyle yinelemeli olarak yeniden boyar. İki çizgenin renk histogramları herhangi bir anda
farklılaşırsa, bu çizgeler izomorf değildir. Hiç farklılaşmazlarsa test sonuçsuz kalır.

Bir mesaj geçişli ÇSA, özet fonksiyonu yerine öğrenilmiş fonksiyonlarla *tam olarak* bunu yapar.
Dolayısıyla teorem (Xu vd., Morris vd.): **bir mesaj geçişli ÇSA iki çizgeyi ancak 1-WL
ayırabiliyorsa ayırabilir.** Klasik başarısızlık örneği 6-döngüye karşı iki ayrık üçgendir: her iki
çizgede de her düğümün derecesi 2'dir ve komşuluklar sonsuza kadar aynı görünür.


In [ ]:
def wl(A, iters=5):
    n = len(A)
    colours = np.zeros(n, dtype=int)
    hist = []
    for _ in range(iters):
        sig = [(colours[v], tuple(sorted(colours[u] for u in np.nonzero(A[v])[0]))) for v in range(n)]
        uniq = {s: i for i, s in enumerate(sorted(set(sig)))}
        colours = np.array([uniq[s] for s in sig])
        hist.append(np.bincount(colours, minlength=n).tolist())
    return colours, hist

def cycle(n):
    A = np.zeros((n, n))
    for i in range(n):
        A[i, (i+1) % n] = A[(i+1) % n, i] = 1
    return A

def two_triangles():
    A = np.zeros((6, 6))
    for base in (0, 3):
        for i in range(3):
            for j in range(3):
                if i != j: A[base+i, base+j] = 1
    return A

G1, G2 = cycle(6), two_triangles()
c1, h1 = wl(G1); c2, h2 = wl(G2)
print("6-döngü          WL renk histogramı:", h1[-1])
print("iki üçgen        WL renk histogramı:", h2[-1])
print("1-WL ikisini ayırabiliyor mu?", h1[-1] != h2[-1])

# Hangi ağırlıkla olursa olsun bir ÇSA ikisine aynı çizge gömmesini verir
def gnn_embed(A, layers=4, d=8, seed=0):
    r = np.random.default_rng(seed)
    H = np.ones((len(A), d))
    for _ in range(layers):
        W1, W2 = r.normal(size=(d, d)), r.normal(size=(d, d))
        H = np.tanh(H@W1 + A@H@W2)
    return H.sum(0)                                   # değişmez okuma

diffs = [np.abs(gnn_embed(G1, seed=s) - gnn_embed(G2, seed=s)).max() for s in range(200)]
print(f"\n200 rastgele ÇSA üzerinde, iki çizge gömmesi arasındaki maks fark: {max(diffs):.2e}")

fig, axes = plt.subplots(1, 3, figsize=(16, 4.2))
for ax, (A, name) in zip(axes[:2], [(G1, "6-döngü"), (G2, "iki üçgen")]):
    th = np.linspace(0, 2*np.pi, len(A), endpoint=False)
    pos = np.c_[np.cos(th), np.sin(th)]
    for u in range(len(A)):
        for v in range(u+1, len(A)):
            if A[u, v]: ax.plot(*zip(pos[u], pos[v]), c="gray", lw=1.5)
    ax.scatter(pos[:, 0], pos[:, 1], s=180, c="steelblue", zorder=3)
    ax.set_title(f"{name}\ntüm dereceler = 2"); ax.set_aspect("equal"); ax.axis("off")

axes[2].hist(diffs, bins=20, color="crimson", alpha=0.8)
axes[2].set_xlabel("maks |gömme(G1) - gömme(G2)|"); axes[2].set_ylabel("sayı")
axes[2].set_title("200 rastgele ÇSA, hepsi çakışıyor")
plt.tight_layout(); plt.show()


### Tavanı aşmak

Üç pratik kaçış yolu; hepsi bir miktar simetriyi ya da hesabı ifade gücüyle takas eder:

1. **Rastgele / benzersiz düğüm öznitelikleri.** Her düğüme rastgele bir vektör verin. Model artık
   yapıları sayabilir ama permütasyona değişmez değildir — değişmezlik çekilişler üzerinden ortalama
   alınarak geri kazanılmalıdır.
2. **Yapısal kodlamalar.** 1-WL'nin türetemeyeceği elle hesaplanmış öznitelikler ekleyin: üçgen
   sayıları, en kısa yol uzaklıkları ya da Laplace özvektörleri (çizge transformer'ları bunu kullanır).
3. **Yüksek mertebeli ÇSA'lar.** Düğümler yerine düğümlerin $k$'lı gruplarıyla mesajlaşın; bu $k$-WL
   ile eşleşir ama $O(n^k)$ maliyetlidir.


In [ ]:
def gnn_embed_rand(A, layers=4, d=8, seed=0):
    r = np.random.default_rng(seed)
    H = r.normal(size=(len(A), d))                    # rastgele düğüm kimlikleri
    for _ in range(layers):
        W1, W2 = r.normal(size=(d, d)), r.normal(size=(d, d))
        H = np.tanh(H@W1 + A@H@W2)
    return np.sort(H.sum(1))                          # sıralı okuma permütasyon değişmezliğini geri kazandırır

def triangle_count(A):  return float(np.trace(A@A@A)/6)
def cycle_features(A):
    return np.array([triangle_count(A), float(np.trace(np.linalg.matrix_power(A, 4))/8),
                     float(np.sort(np.linalg.eigvalsh(A))[0])])

print(f"{'yöntem':34s} {'G1 ile G2 ayrılıyor mu?':>24s}")
print(f"{'düz ÇSA (sabit öznitelik)':34s} {str(max(diffs) > 1e-6):>26s}")
d_rand = np.mean([np.abs(gnn_embed_rand(G1, seed=s)-gnn_embed_rand(G2, seed=s)).max() for s in range(50)])
print(f"{'ÇSA + rastgele düğüm özniteliği':34s} {str(d_rand > 1e-6):>26s}")
print(f"{'yapısal kodlamalar':34s} {str(not np.allclose(cycle_features(G1), cycle_features(G2))):>26s}")
print(f"\nüçgen sayıları: 6-döngü = {triangle_count(G1):.0f},  iki üçgen = {triangle_count(G2):.0f}")
print("Çizge spektrumları:", np.round(np.linalg.eigvalsh(G1), 2), "vs", np.round(np.linalg.eigvalsh(G2), 2))


## 5. Aşırı Düzleşme ve Aşırı Sıkışma

Derin ÇSA'ların başarısız olmasının iki ayrı nedeni.

**Aşırı düzleşme (oversmoothing).** Tekrarlanan komşuluk ortalaması bir difüzyon sürecidir: düğüm
öznitelikleri yalnızca dereceye bağlı bir fonksiyona yakınsar ve ayırt edici bilginin tamamı
kaybolur. **Dirichlet enerjisi**
$E(H) = \frac{1}{2}\sum_{(u,v)\in E}\lVert h_u - h_v\rVert^2$ derinlikle geometrik olarak azalır ve
azalma hızını spektral aralık belirler.

**Aşırı sıkışma (oversquashing).** Üstel büyüyen $k$-atlamalı bir komşuluktan gelen bilgi sabit
boyutlu bir vektöre sıkıştırılmak zorundadır. Çizgede bir darboğaz varsa uzak bir $u$ için Jacobian
$\partial h_v^{(k)}/\partial x_u$ yok denecek kadar küçülür — model sinyali nasıl eğitilirse eğitilsin
*iletemez*. Artık bağlantılar ve normalleştirme aşırı düzleşmeyi geciktirir; aşırı sıkışma ise
topolojinin değişmesini gerektirir (yeniden bağlama, sanal global düğümler, çizge transformer'ları).


In [ ]:
def dirichlet(A, H):
    idx = np.transpose(np.nonzero(np.triu(A)))
    return 0.5*sum(np.sum((H[u]-H[v])**2) for u, v in idx)

def make_graph(kind, n=60, seed=0):
    r = np.random.default_rng(seed)
    A = np.zeros((n, n))
    if kind == "rastgele":
        A = (r.random((n, n)) < 0.08).astype(float); A = np.triu(A, 1); A = A + A.T
    else:                                            # TEK bir kenarla birleştirilmiş iki klik: bir darboğaz
        h = n//2
        A[:h, :h] = 1; A[h:, h:] = 1
        np.fill_diagonal(A, 0); A[h-1, h] = A[h, h-1] = 1
    return A

fig, axes = plt.subplots(1, 3, figsize=(16, 4.2))
for kind, c in [("rastgele", "steelblue"), ("darboğaz", "crimson")]:
    A = make_graph(kind)
    D = np.diag(1/np.maximum(A.sum(1), 1))
    H = np.random.default_rng(0).normal(size=(len(A), 16))
    e = [dirichlet(A, H)]
    for _ in range(30):
        H = D@A@H                                    # saf ortalama toplayıcı
        e.append(dirichlet(A, H))
    axes[0].semilogy(np.array(e)/e[0], lw=2, c=c, label=kind)
axes[0].set_xlabel("katman"); axes[0].set_ylabel("Dirichlet enerjisi (normalleştirilmiş, log)")
axes[0].set_title("Aşırı düzleşme: öznitelikler sabite doğru çöküyor")
axes[0].legend(fontsize=9); axes[0].grid(alpha=0.3, which="both")

# Aşırı sıkışma: bir düğümün darboğazın ötesindeki uzak bir düğüme duyarlılığı
A = make_graph("darboğaz"); n = len(A)
D = np.diag(1/np.maximum(A.sum(1), 1)); Ahat = D@A
sens = []
for k in range(1, 13):
    J = np.linalg.matrix_power(Ahat, k)              # doğrusal bir ÇSA için d h_v^(k) / d x_u
    sens.append([J[0, 1], J[0, n-1]])                # aynı klik ile darboğazın ötesi
sens = np.array(sens)
axes[1].semilogy(range(1, 13), sens[:, 0], "o-", lw=2, label="aynı küme içinde")
axes[1].semilogy(range(1, 13), sens[:, 1], "s-", lw=2, label="darboğazın ötesinde")
axes[1].set_xlabel("katman sayısı k"); axes[1].set_ylabel("|d h_v / d x_u|  (log)")
axes[1].set_title("Aşırı sıkışma: darboğaz sinyali söndürüyor")
axes[1].legend(fontsize=9); axes[1].grid(alpha=0.3, which="both")

axes[2].imshow(A, cmap="Greys")
axes[2].set_title("Tek bir kenarla birleşmiş iki küme"); axes[2].set_xticks([]); axes[2].set_yticks([])
plt.tight_layout(); plt.show()
print(f"8 katmandan sonra darboğaz ötesi duyarlılık, küme içi duyarlılığın {sens[7,1]/sens[7,0]:.1e} katı.")


## 6. Transformer'lar Tam Çizge Üzerindeki ÇSA'lardır

Öz-dikkat, her düğümün diğer her düğümün komşusu olduğu ve kenar ağırlığının uç noktalardan
öğrenildiği bir mesaj geçişidir. Bu bakış birkaç tanıdık olguyu aynı anda açıklar:

- Dikkatte **aşırı sıkışma sorunu yoktur** — her çift bir atlama uzaklıktadır; çizge
  transformer'larının uzun menzilli görevler için cazip olmasının nedeni budur.
- Buna karşılık hiçbir çizge yapısı da yoktur; yapı **konumsal / yapısal kodlama** olarak yeniden
  enjekte edilmelidir: Laplace özvektörleri, rastgele yürüyüş dönüş olasılıkları ya da dikkat sapması
  olarak kullanılan en kısa yol uzaklıkları.
- Dikkatin permütasyon eşdeğerliği *küme* simetrisidir; konum kodlamaları bunu tam olarak alanın
  gerektirdiği biçimde kasten bozar.

Bedeli her zamanki gibidir: $O(|E|)$ yerine $O(n^2)$; seyrek bir çizge için bu büyük bir sıçramadır.


In [ ]:
# Rastgele yürüyüş yapısal kodlaması: k adım sonra dönüş olasılığı yerel yapının parmak izidir
def rw_encoding(A, K=8):
    D = np.diag(1/np.maximum(A.sum(1), 1)); P = D@A
    M, feats = np.eye(len(A)), []
    for _ in range(K):
        M = M@P
        feats.append(np.diag(M).copy())
    return np.array(feats).T                          # (n, K)

G1e, G2e = rw_encoding(cycle(6)), rw_encoding(two_triangles())
print("rastgele yürüyüş kodlaması, 6-döngünün 0. düğümü :", G1e[0].round(3))
print("rastgele yürüyüş kodlaması, iki üçgenin 0. düğümü:", G2e[0].round(3))
print("ayırt edilebilir mi?", not np.allclose(G1e[0], G2e[0]))

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
A_sparse = make_graph("rastgele", n=40)
axes[0].imshow(A_sparse, cmap="Greys"); axes[0].set_title("ÇSA: mesajlar yalnızca kenarlar boyunca")
axes[1].imshow(np.ones_like(A_sparse), cmap="Greys", vmin=0, vmax=1.4)
axes[1].set_title("Transformer: tam çizge, öğrenilmiş ağırlıklar")
for ax in axes[:2]: ax.set_xticks([]); ax.set_yticks([])

ns = np.logspace(1, 4, 100); avg_deg = 6
axes[2].loglog(ns, ns*avg_deg, lw=2, label="ÇSA maliyeti ~ |E|")
axes[2].loglog(ns, ns**2, lw=2, label="transformer maliyeti ~ n^2")
axes[2].set_xlabel("düğüm sayısı"); axes[2].set_ylabel("katman başına mesaj")
axes[2].set_title("Tam çizgenin bedeli"); axes[2].legend(fontsize=9)
axes[2].grid(alpha=0.3, which="both")
plt.tight_layout(); plt.show()


## 7. Özet

| Kavram | Açıklama |
|---|---|
| **Değişmezlik / eşdeğerlik** | $f(gx)=f(x)$ ile $f(gx)=gf(x)$; eşdeğerli katmanları yığ, bir kez havuzla |
| **Geometrik kurgu** | ESA, ÇSA, transformer, deep sets = tek tarif, farklı gruplar |
| **Artırım mı mimari mi** | Yaklaşık ve veriye bağlı, karşısında tam ve her yerde geçerli |
| **Mesaj geçişi** | $h_v \leftarrow \phi(h_v, \bigoplus_{u\in\mathcal{N}(v)}\psi(h_v,h_u))$ |
| **Toplayıcı seçimi** | Toplam çoklu kümelerde birebirdir; ortalama ve maksimum değildir |
| **1-WL tavanı** | Mesaj geçişli ÇSA'lar 1-WL'nin ayıramadığını ayıramaz |
| **Tavanı aşmak** | Rastgele öznitelikler, yapısal kodlamalar veya $O(n^k)$ maliyetli $k$-mertebeli ÇSA'lar |
| **Aşırı düzleşme** | Dirichlet enerjisi derinlikle geometrik olarak azalır |
| **Aşırı sıkışma** | Darboğazlar uzak düğüm Jacobianlarını yok eder; çözüm derinlik değil yeniden bağlamadır |
| **Çizge transformer'ları** | Tam çizge dikkati + yapısal kodlamalar; $O(n^2)$ |

**Sonraki Defter →** Yorumlanabilirlik ve Mekanistik Analiz
